# GRM shard batch submission (Google Batch / dsub)

Moves GRM shard construction off the interactive VM and onto Google Batch
-- each Batch task gets its own dedicated VM, avoiding the inter-shard CPU
and memory-bandwidth contention seen running many concurrent plink
processes on one machine locally (see `grm_shard_run.ipynb`). Stages:

1. **Troubleshooting job** -- the smallest possible `dsub` submission
   (no plink, no real inputs), just to confirm submission itself works
   (auth, project/region config, logging bucket permissions, the
   `google-batch` provider) before anything plink-specific can go wrong
   and get confused with dsub-specific problems.
2. **Validation batch** -- `N_SHARDS=16` (few, fat shards -- fixed
   per-shard cost doesn't shrink with more shards, it's just paid more
   times), grouped into `N_TASKS=4` Batch tasks (multiple shards/task),
   amortizing the real ~15 min/task panel-download overhead measured in
   an earlier one-shard/task attempt across more actual work per task.
3. **Full run** -- not built yet. Comes after (2) confirms real
   timing/cost at this shard size and task grouping.

**Everything here started as a first draft and has been fixed against
several real platform issues since** -- see the Status cell below for
what's actually been confirmed working.

## Prerequisites

`dsub` submits to Google Batch from wherever this cell runs (this
notebook's VM), but the actual work happens on separate Batch-provisioned
machines -- neither the interactive VM's compute resource nor its plink
install matter for what follows. Needs `dsub` installed and `gcloud`
already authenticated to the right project (both should already be true on
a Verily Workbench VM, but confirmed explicitly below rather than assumed).

In [ ]:
%%bash
set -e

if ! command -v dsub >/dev/null 2>&1; then
  pip install --quiet dsub
fi
dsub --version

echo "--- gcloud config ---"
gcloud config list --format='text(core.project,compute.region)' 2>&1 || true

## Inputs

Fill in `PROJECT_ID`/`REGION` from the `gcloud config` output above if they
didn't print automatically. `MEMORY_MB` is a placeholder -- replace with
the real `max_rss_gb` measured in `grm_shard_timing.ipynb` (with
`--read-freq`/`--memory` applied) plus ~20-30% headroom before running the
validation batch for real; the troubleshooting job below doesn't use it.

In [ ]:
import os
import math

PROJECT_ID = "wb-fulgent-almond-9474"
REGION = "us-central1"   # TODO -- confirm against gcloud config output above

# dsub's default (project's Compute Engine default SA) failed with PERMISSION_DENIED --
# the calling identity doesn't have actAs rights on it. Using this VM's own attached
# "pet" SA instead, which the caller can already act as without an extra IAM grant.
SERVICE_ACCOUNT = "pet-27799165194323faf22e2@wb-fulgent-almond-9474.iam.gserviceaccount.com"

# VPC Service Controls is enabled on this project -- Batch jobs must run with no
# external IP, on the project's private network/subnet. Per Verily Workbench's own
# dsub guide (support.workbench.verily.com/docs/guides/workflows/dsub/), the network
# and subnetwork are literally named "network"/"subnetwork" in every Workbench project.
NETWORK = f"projects/{PROJECT_ID}/global/networks/network"
SUBNETWORK = f"projects/{PROJECT_ID}/regions/{REGION}/subnetworks/subnetwork"

WORKSPACE_BUCKET = os.path.expanduser(
    "~/workspace/Data from All of Us Controlled Tier /shared-env-pilot"
)
# confirmed via `ps -eo pid,args | grep gcsfuse` -- this local mount's gcsfuse process
# is `gcsfuse ... shared-env-pilot-wb-fulgent-almond-9474 <this path>`, i.e. the bucket
# ROOT maps directly to this mount point, no extra path suffix. The earlier guess
# ("gs://wb-fulgent-almond-9474/shared-env-pilot") used the project ID as the bucket
# name, which doesn't exist (404) -- that's what caused the troubleshooting job's log
# upload, and therefore the job itself, to fail.
WORKSPACE_BUCKET_GS = "gs://shared-env-pilot-wb-fulgent-almond-9474"

BUCKET_DIR = f"{WORKSPACE_BUCKET}/data/03_grm_shards"
BUCKET_DIR_GS = f"{WORKSPACE_BUCKET_GS}/data/03_grm_shards"

GRM_INPUT_DIR = f"{BUCKET_DIR}/grm_input"
GRM_INPUT_DIR_GS = f"{BUCKET_DIR_GS}/grm_input"

BED_NAME = "genome_wide_round2b_thinned_bed"   # from genome_wide_qc_thinning_merge.ipynb
BED_PREFIX = f"{BUCKET_DIR}/{BED_NAME}"
BED_PREFIX_GS = f"{BUCKET_DIR_GS}/{BED_NAME}"   # gs:// form -- needed for a server-side
                                                 # (bucket-to-bucket) copy, see the staging cell

# the default Batch image (ubuntu2204) has neither wget nor curl (confirmed via real
# task logs: both "command not found") -- rather than guess at a third download tool
# or fight with whether this no-external-IP VM can even reach s3.amazonaws.com at all
# (VPC-SC + Private Google Access covers Google APIs, not arbitrary internet hosts),
# stage plink into the bucket once from here and have dsub localize it like any other
# input via gsutil, which is already proven to work (BED_DIR staging succeeded).
PLINK_BIN_GS = f"{BUCKET_DIR_GS}/bin/plink"

# The first validation batch (one shard per task, N_SHARDS=300) measured real per-task
# overhead: ~15 min just for the BED_DIR download (gsutil rsync of the ~49 GB panel),
# BEFORE any plink compute even starts -- a fixed cost paid once per TASK, independent
# of how much work that task then does. At one shard/task that tax dominated. Since
# each Batch task gets a dedicated VM (no inter-shard contention, unlike the local
# concurrent run), bumping threads per task is free of the contention problems seen
# locally -- so this run uses fewer, fatter shards (N_SHARDS=16, see Stage 2) with
# MULTIPLE shards processed per task, amortizing that fixed tax across more real work
# instead of paying it once per shard.
MACHINE_VCPUS = 16

# sized the same way as grm_shard_run.ipynb: 2x bed file size + 4 GB headroom, not a
# guess -- update BED_SIZE_GB below if the bed file changes. N1 custom machine types
# require memory to be an EXACT multiple of 256 MB -- the raw formula's output
# (103628) isn't one, which Batch rejected outright ("Instance properties must
# provide existing machine type", CODE_GCE_BAD_REQUEST) before any task could even
# start. Round up to the nearest 256 MB to guarantee a valid value.
BED_SIZE_GB = 48.6
_raw_memory_mb = (BED_SIZE_GB * 2 + 4) * 1024
MEMORY_MB = math.ceil(_raw_memory_mb / 256) * 256

print(BUCKET_DIR_GS)
print(f"MACHINE_VCPUS={MACHINE_VCPUS}, MEMORY_MB={MEMORY_MB}")

## Stage a clean input folder (one-time)

Each `dsub` task downloads whatever's in this folder -- keep it to just the
4 files plink actually needs, not the whole `03_grm_shards/` bucket
contents (which also has the per-chromosome intermediates, logs, etc.).

In [ ]:
%%bash -s "$BED_PREFIX_GS" "$GRM_INPUT_DIR_GS"
set -e
BED_PREFIX_GS=$1
GRM_INPUT_DIR_GS=$2

# server-side (bucket-to-bucket) copies via gs:// URIs -- NOT `cp` between two
# gcsfuse-mounted paths. gcsfuse has no server-side-copy concept: a plain `cp` between
# two mounted paths reads the whole object down to this VM's local disk and re-uploads
# it, even when source and dest are in the same bucket. For the ~49 GB bed file that's
# both slow and can exhaust local disk ("No space left on device") for no reason --
# `gcloud storage cp` on gs:// paths does this as a metadata-only operation on GCS's
# side, touching zero local disk.
#
# Actual sizes are compared (via `gcloud storage du`) rather than trusting
# `--no-clobber` blindly -- a partial/truncated object can be left at the destination
# from an earlier failed copy (this happened once already: plink rejected the .bed
# with "Invalid .bed file size" because a stale truncated copy from the original
# disk-space failure was silently skipped by --no-clobber).
BED_NAME=$(basename "$BED_PREFIX_GS")
for ext in bed bim fam; do
  src="${BED_PREFIX_GS}.${ext}"
  dest="${GRM_INPUT_DIR_GS}/${BED_NAME}.${ext}"
  src_size=$(gcloud storage du "$src" 2>/dev/null | awk '{print $1}')
  dest_size=$(gcloud storage du "$dest" 2>/dev/null | awk '{print $1}')
  if [ -n "$dest_size" ] && [ "$src_size" = "$dest_size" ]; then
    echo "skip ${ext}: already staged, same size ($dest_size bytes)"
  else
    echo "copying ${ext} (src=${src_size:-?} bytes, dest=${dest_size:-missing})"
    gcloud storage cp "$src" "$dest"
  fi
done

freq_src="${BED_PREFIX_GS}_freq.frq"
freq_dest="${GRM_INPUT_DIR_GS}/${BED_NAME}_freq.frq"

# the .frq file is only ever computed in local scratch (grm_shard_timing.ipynb /
# grm_shard_run.ipynb), never persisted to the bucket on its own -- if it's missing at
# the bucket path, upload it from local scratch first rather than failing here. This
# direction (local -> gs://) is a real upload, but the file is tiny, so it's fine.
if ! gcloud storage ls "$freq_src" >/dev/null 2>&1; then
  local_freq="$HOME/scratch_grm/${BED_NAME}_freq.frq"
  if [ -f "$local_freq" ]; then
    echo "bucket copy of .frq missing -- uploading from local scratch instead"
    gcloud storage cp "$local_freq" "$freq_src"
  else
    echo "no .frq file found at bucket path or local scratch ($local_freq) -- run grm_shard_timing.ipynb's precompute cell first" >&2
    exit 1
  fi
fi

freq_src_size=$(gcloud storage du "$freq_src" 2>/dev/null | awk '{print $1}')
freq_dest_size=$(gcloud storage du "$freq_dest" 2>/dev/null | awk '{print $1}')
if [ -n "$freq_dest_size" ] && [ "$freq_src_size" = "$freq_dest_size" ]; then
  echo "skip freq: already staged, same size ($freq_dest_size bytes)"
else
  gcloud storage cp "$freq_src" "$freq_dest"
fi

gcloud storage ls -l "$GRM_INPUT_DIR_GS"

### Stage the plink binary too (one-time)

The Batch worker image has neither `wget` nor `curl`, so the task can't
download plink itself. Upload the same binary used by
`grm_shard_timing.ipynb`/`grm_shard_run.ipynb` (`~/bin/plink`) to the
bucket once, and let `dsub` localize it via `--input` the same way it
already localizes `BED_DIR` -- that path is proven to work.

In [ ]:
%%bash
set -e

BIN_DIR="$HOME/bin"
mkdir -p "$BIN_DIR"

if [ ! -x "$BIN_DIR/plink" ]; then
  # same install as grm_shard_timing.ipynb/grm_shard_run.ipynb -- URL is dated; if it
  # 404s, get the current link from https://www.cog-genomics.org/plink/1.9/
  PLINK_URL="https://s3.amazonaws.com/plink1-assets/plink_linux_x86_64_20230116.zip"
  cd /tmp
  wget -q -O plink.zip "$PLINK_URL"
  unzip -o -q plink.zip plink -d "$BIN_DIR"
  chmod +x "$BIN_DIR/plink"
fi

"$BIN_DIR/plink" --version

In [ ]:
%%bash -s "$PLINK_BIN_GS"
set -e
PLINK_BIN_GS=$1

local_plink="$HOME/bin/plink"
if [ ! -x "$local_plink" ]; then
  echo "no local plink at $local_plink -- run grm_shard_timing.ipynb's or grm_shard_run.ipynb's Setup cell first" >&2
  exit 1
fi

gcloud storage cp "$local_plink" "$PLINK_BIN_GS"
gcloud storage ls -l "$PLINK_BIN_GS"

## Stage 1 -- troubleshooting job

The smallest possible submission: no real inputs, no plink, just
`echo`/`hostname`/`date` on the cheapest default machine. If this doesn't
come back `SUCCESS` via `dstat` below, the problem is in dsub/Batch/auth
setup, not in anything plink- or GRM-specific -- worth fully resolving
before moving to Stage 2.

In [ ]:
%%bash -s "$PROJECT_ID" "$REGION" "$BUCKET_DIR_GS" "$SERVICE_ACCOUNT" "$NETWORK" "$SUBNETWORK"
set -e
PROJECT_ID=$1
REGION=$2
BUCKET_DIR_GS=$3
SERVICE_ACCOUNT=$4
NETWORK=$5
SUBNETWORK=$6

dsub \
  --provider google-batch \
  --project "$PROJECT_ID" \
  --regions "$REGION" \
  --logging "${BUCKET_DIR_GS}/dsub_logs" \
  --service-account "$SERVICE_ACCOUNT" \
  --network "$NETWORK" \
  --subnetwork "$SUBNETWORK" \
  --use-private-address \
  --name "grm-troubleshoot" \
  --command 'echo "hello from dsub"; hostname; date' \
  > /tmp/troubleshoot_job_id.txt

cat /tmp/troubleshoot_job_id.txt

In [ ]:
%%bash -s "$PROJECT_ID" "$REGION"
set -e
PROJECT_ID=$1
REGION=$2
JOB_ID=$(tail -1 /tmp/troubleshoot_job_id.txt)

dstat --provider google-batch --project "$PROJECT_ID" --location "$REGION" --jobs "$JOB_ID" --users 'jupyter' --status '*'

## Stage 2 -- validation batch (few, fat shards, multiple per task)

Only run this once Stage 1 comes back `SUCCESS`. `N_SHARDS=16` (down from
the 300 tried in the first validation batch) -- per the "fewer, larger
shards" direction, since per-shard fixed cost (loading the panel) doesn't
shrink with more shards, it just gets paid more times. `N_TASKS=4` groups
those 16 shards into 4 Batch tasks (4 shards/task), so the real ~15 min/task
`BED_DIR`-download overhead measured in the first validation batch gets
amortized across 4x more work instead of being paid once per shard.

`k` values are assigned round-robin across tasks (task `i` gets `k = i,
i+N_TASKS, i+2*N_TASKS, ...`) rather than contiguous blocks -- plink's
`--parallel` balances shards by off-diagonal *pair* count, not row count,
so different `k` can have somewhat different real costs; round-robin
spreads any such difference across all tasks instead of concentrating it
in one.

In [ ]:
N_SHARDS = 16   # fewer, fatter shards -- fixed per-shard cost doesn't shrink with more
                # shards, it's just paid more times
N_TASKS = 4     # 4 shards/task on average -- amortizes the ~15 min/task BED_DIR download
                # overhead measured in the first (1-shard/task) validation batch

task_shards = {i: [] for i in range(N_TASKS)}
for k in range(1, N_SHARDS + 1):
    task_shards[(k - 1) % N_TASKS].append(k)

TASKS_PATH = "/tmp/grm_validation_tasks.tsv"
with open(TASKS_PATH, "w") as f:
    f.write("--env K_LIST\t--env N_SHARDS\n")
    for i in range(N_TASKS):
        k_list = ",".join(str(k) for k in task_shards[i])
        f.write(f"{k_list}\t{N_SHARDS}\n")

print(open(TASKS_PATH).read())
print(f"{N_TASKS} tasks, {N_SHARDS} total shards, {N_SHARDS / N_TASKS:.1f} shards/task on average")

In [ ]:
%%bash -s "$PROJECT_ID" "$REGION" "$BUCKET_DIR_GS" "$GRM_INPUT_DIR_GS" "$PLINK_BIN_GS" "$MACHINE_VCPUS" "$MEMORY_MB" "$SERVICE_ACCOUNT" "$NETWORK" "$SUBNETWORK"
set -e
PROJECT_ID=$1
REGION=$2
BUCKET_DIR_GS=$3
GRM_INPUT_DIR_GS=$4
PLINK_BIN_GS=$5
MACHINE_VCPUS=$6
MEMORY_MB=$7
SERVICE_ACCOUNT=$8
NETWORK=$9
SUBNETWORK=${10}

# N1 custom machine types need an "-ext" suffix once memory exceeds 8 GiB/vCPU
# (8192 MB/vCPU) -- without it the machine-type string is invalid and Batch rejects
# the VM at provisioning time, before the task (or its log) ever runs. That's almost
# certainly why the last attempt's 5 tasks all failed with zero logs uploaded at all.
MEM_PER_VCPU=$((MEMORY_MB / MACHINE_VCPUS))
if [ "$MEM_PER_VCPU" -gt 8192 ]; then
  MACHINE_TYPE="n1-custom-${MACHINE_VCPUS}-${MEMORY_MB}-ext"
else
  MACHINE_TYPE="n1-custom-${MACHINE_VCPUS}-${MEMORY_MB}"
fi
echo "machine-type: $MACHINE_TYPE ($MEM_PER_VCPU MB/vCPU)"

dsub \
  --provider google-batch \
  --project "$PROJECT_ID" \
  --regions "$REGION" \
  --logging "${BUCKET_DIR_GS}/dsub_logs" \
  --service-account "$SERVICE_ACCOUNT" \
  --network "$NETWORK" \
  --subnetwork "$SUBNETWORK" \
  --use-private-address \
  --name "grm-shard-validation" \
  --machine-type "$MACHINE_TYPE" \
  --disk-size 150 \
  --input-recursive BED_DIR="$GRM_INPUT_DIR_GS" \
  --input PLINK_BIN="$PLINK_BIN_GS" \
  --output-recursive OUT_DIR="${BUCKET_DIR_GS}/shards" \
  --command '
    set -e
    chmod +x "$PLINK_BIN"

    BED_PREFIX="${BED_DIR}/genome_wide_round2b_thinned_bed"
    FREQ_PATH="${BED_DIR}/genome_wide_round2b_thinned_bed_freq.frq"

    # BED_DIR is downloaded once by dsub before this script runs -- reused across every
    # shard in K_LIST below, instead of paying the ~15 min download again per shard
    IFS="," read -ra KS <<< "$K_LIST"
    for K in "${KS[@]}"; do
      echo "=== shard $K of $N_SHARDS ==="
      "$PLINK_BIN" \
        --bfile "$BED_PREFIX" \
        --make-grm-bin \
        --parallel "$K" "${N_SHARDS}" \
        --read-freq "$FREQ_PATH" \
        --memory '"$MEMORY_MB"' \
        --threads '"$MACHINE_VCPUS"' \
        --out "${OUT_DIR}/grm_shard_${K}_of_${N_SHARDS}"
      # .grm.N.bin (pairwise valid-genotype counts) isn'\''t needed downstream --
      # grm_shard_tool only reads .grm.bin -- drop it before upload to halve output size
      rm -f "${OUT_DIR}/grm_shard_${K}_of_${N_SHARDS}.grm.N.bin.${K}"
    done
  ' \
  --tasks /tmp/grm_validation_tasks.tsv \
  > /tmp/validation_job_id.txt

cat /tmp/validation_job_id.txt

In [ ]:
%%bash -s "$PROJECT_ID" "$REGION"
set -e
PROJECT_ID=$1
REGION=$2
JOB_ID=$(tail -1 /tmp/validation_job_id.txt)

dstat --provider google-batch --project "$PROJECT_ID" --location "$REGION" --jobs "$JOB_ID" --users 'jupyter' --status '*'

## Validation results

Per-task status and wall-clock (via `dstat --full --format json`, parsed
so create-time/end-time can be diffed), plus a listing of the actual
output files -- confirms real `SUCCESS` (not just "no client-side error"),
checks output files exist and aren't suspiciously small, and gives the
real Batch-including-provisioning wall-clock to compare against the
isolated interactive timing (~958.6s at `--threads 12`, no VM
startup/staging overhead).

In [ ]:
import json
import subprocess
from datetime import datetime

JOB_ID = open("/tmp/validation_job_id.txt").read().strip()

result = subprocess.run(
    ["dstat", "--provider", "google-batch", "--project", PROJECT_ID,
     "--location", REGION, "--jobs", JOB_ID, "--users", "jupyter",
     "--status", "*", "--full", "--format", "json"],
    capture_output=True, text=True, check=True,
)
tasks = json.loads(result.stdout)

def parse_ts(ts):
    return datetime.fromisoformat(ts.replace("Z", "+00:00")) if ts else None

print(f"{'task':>4}  {'status':<10} {'wall_clock_sec':>14}  {'k_list'}")
rows = []
for t in sorted(tasks, key=lambda t: int(t.get("task-id", 0))):
    create = parse_ts(t.get("create-time"))
    end = parse_ts(t.get("end-time"))
    wall_sec = (end - create).total_seconds() if create and end else None
    k_list = t.get("envs", {}).get("K_LIST", "?")
    rows.append((t.get("task-id"), t.get("status"), wall_sec, k_list))
    wall_str = f"{wall_sec:.1f}" if wall_sec is not None else "N/A"
    print(f"{t.get('task-id'):>4}  {t.get('status'):<10} {wall_str:>14}  {k_list}")

n_success = sum(1 for _, status, _, _ in rows if status == "SUCCESS")
print(f"\n{n_success}/{len(rows)} tasks SUCCESS")

wall_times = [w for _, _, w, _ in rows if w is not None]
if wall_times:
    print(f"wall-clock range: {min(wall_times):.1f}s - {max(wall_times):.1f}s "
          f"(spread {100 * (max(wall_times) - min(wall_times)) / (sum(wall_times) / len(wall_times)):.1f}%)")
    # this run (few, fat shards, multiple/task) has no prior isolated baseline to compare
    # against -- unlike the first (1-shard/task, N_SHARDS=300) validation batch, this is
    # itself the first real timing data at this shard size/machine config, not a check
    # against a known number. Divide wall-clock by shards-per-task (K_LIST length) for a
    # rough per-shard figure, keeping in mind the first shard in each task still pays the
    # ~15 min BED_DIR download that later shards in the same task don't.

In [ ]:
%%bash -s "$BUCKET_DIR_GS"
set -e
BUCKET_DIR_GS=$1

echo "--- output files ---"
gcloud storage ls -l "${BUCKET_DIR_GS}/shards/" | sort -k3

In [ ]:
%%bash -s "$BUCKET_DIR_GS"
set -e
BUCKET_DIR_GS=$1

# read the FULL file content, not `tail -1` -- for a --tasks (multi-task) submission,
# dsub's stdout doesn't necessarily end with a bare job-id on its own line the way a
# single-task job's does, so `tail -1` can silently grab the wrong string (this bit us
# once already: a dstat --jobs query with a tail-1-derived id came back empty `[]`).
JOB_ID=$(cat /tmp/validation_job_id.txt)
echo "JOB_ID: $JOB_ID"
echo

LOG_PREFIX="${BUCKET_DIR_GS}/dsub_logs/${JOB_ID}"
echo "--- log files matching ${LOG_PREFIX}* ---"
gcloud storage ls "${LOG_PREFIX}"'*' || echo "(none found -- job likely failed before it could write any log, e.g. a provisioning-time error; check dstat --full's status-detail/events instead)"

echo
for f in $(gcloud storage ls "${LOG_PREFIX}"'*' 2>/dev/null); do
  echo
  echo "=== $f ==="
  gcloud storage cat "$f"
done

## Status

**Stage 1 (troubleshooting job) confirmed `SUCCESS`** on 2026-07-17
(16:47:34), via `dstat`. Three real platform issues were found and fixed
along the way, none plink-specific -- exactly what this stage exists to
catch early, before risking anything on a real shard:

1. `PERMISSION_DENIED: ... act as service account ...` -- dsub's default
   (the project's Compute Engine default SA) isn't one the calling
   identity can act as. Fixed by passing `--service-account` with this
   VM's own attached pet SA instead.
2. `no_external_ip_address field is invalid ... VPC Service Controls is
   enabled` -- Batch jobs on this project must run with no external IP,
   on the project's private network/subnet. Fixed by adding
   `--network`/`--subnetwork` (Verily Workbench's fixed `network`/
   `subnetwork` resource names) and `--use-private-address`, per
   Workbench's own dsub guide.
3. `WORKSPACE_BUCKET_GS` was a guess (`gs://<project-id>/...`) that
   pointed at a bucket that doesn't exist (404) -- this, not Private
   Google Access, turned out to be why the job ran but then failed
   (couldn't upload its own log). Fixed by reading the real bucket name
   directly off the gcsfuse mount (`ps -eo pid,args | grep gcsfuse`):
   `shared-env-pilot-wb-fulgent-almond-9474`.

## Next steps

1. Run Stage 2 (validation batch, 5 shards) -- same fixes are wired into
   that `dsub` call. `MACHINE_VCPUS` is now 12 (bumped from an earlier,
   too-conservative guess of 6 -- see the Inputs cell's comment for the
   real timing data behind that), and `MEMORY_MB` is sized from the real
   bed-file-size formula.
2. Compare Stage 2's real per-task wall-clock/cost against the
   interactive `grm_shard_timing.ipynb`/`grm_shard_run.ipynb` numbers --
   Batch tasks have extra overhead (VM provisioning, downloading plink +
   inputs fresh each time) the interactive timing didn't capture, worth
   knowing before extrapolating cost for the full run.
3. Once (1)-(2) look sane and `N_SHARDS` is settled, scale `VALIDATION_KS`
   up to the full `range(1, N_SHARDS + 1)` for the real run -- not built
   as its own cell yet, deliberately, until the above is confirmed.